In [24]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
import joblib # для збереження моделі

In [28]:
# Комірка 2-3 (Об'єднана)
categories = ["Season", "Time", "Weather"]
models = {} # Тут будемо зберігати навчені моделі

# Визначаємо колонки
numeric_features = ['BPM', 'Energy', 'Dance', 'Acoustic', 'Valence']
categorical_features = ['Key'] 

for cat in categories:
    print(f"\n🚀 Тренуємо модель для: {cat}")
    
    # Завантажуємо датасет
    df = pd.read_csv(f"{cat.lower()}.csv")
    
    # Видаляємо рядки з пропусками
    df = df.dropna(subset=numeric_features + categorical_features + [f'{cat}_Label'])
    
    X = df[numeric_features + categorical_features]
    y = df[f'{cat}_Label']
    
    # Розбиваємо на трен/тест
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # 🔥 ОСЬ ФІКС: Створюємо НОВИЙ препроцесор для КОЖНОЇ категорії окремо! 🔥
    numeric_transformer = StandardScaler()
    categorical_transformer = OneHotEncoder(handle_unknown='ignore')
    
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_features),
            ('cat', categorical_transformer, categorical_features)
        ])
    
    # Створюємо повний пайплайн
    clf = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42))
    ])
    
    # Навчаємо
    clf.fit(X_train, y_train)
    
    # Оцінюємо
    score = clf.score(X_test, y_test)
    print(f"📊 Точність ({cat}): {score:.2f}")
    
    # Зберігаємо готову незалежну модель
    models[cat] = clf


🚀 Тренуємо модель для: Season
📊 Точність (Season): 0.46

🚀 Тренуємо модель для: Time
📊 Точність (Time): 0.42

🚀 Тренуємо модель для: Weather
📊 Точність (Weather): 0.46


In [29]:
# Твій тестовий трек
sample_track = {
    'Song': 'Test Summer Hit',
    'Artist': 'Test Artist',
    'BPM': 125,
    'Energy': 0.85,
    'Dance': 0.75,
    'Acoustic': 0.05,
    'Valence': 0.90,
    'Key': 'C Major'
}

print(f"Аналізуємо трек: {sample_track['Song']} - {sample_track['Artist']}")
print("-" * 30)

# Перетворюємо словник на DataFrame в 1 рядок
input_df = pd.DataFrame([sample_track])

for cat in categories:
    model = models[cat]
    
    # Отримуємо всі можливі класи для цієї моделі (напр. ['rainy', 'sunny', ...])
    classes = model.classes_
    
    # Отримуємо ймовірності у відсотках
    probabilities = model.predict_proba(input_df)[0]
    
    print(f"\n[{cat.upper()}] Профіль:")
    
    # Сортуємо результати від найбільшого до найменшого
    results = list(zip(classes, probabilities))
    results.sort(key=lambda x: x[1], reverse=True)
    
    for label, prob in results:
        print(f"  - {label}: {prob * 100:.1f}%")

Аналізуємо трек: Test Summer Hit - Test Artist
------------------------------

[SEASON] Профіль:
  - winter: 41.0%
  - autumn: 31.0%
  - spring: 18.0%
  - summer: 10.0%

[TIME] Профіль:
  - night: 83.0%
  - morning: 9.0%
  - evening: 5.0%
  - afternoon: 3.0%

[WEATHER] Профіль:
  - rainy: 76.0%
  - snowy: 12.0%
  - sunny: 8.0%
  - cloudy: 4.0%
